# Analysing a joint dataset

This notebook runs an end-to-end joint analysis of the primary CMB (MFLike TT/TE/EE) and the
CMB-lensing reconstruction:

1. build the joint likelihood and evaluate it at the fiducial point;
2. fold in the **cross-covariance** between the two probes;
3. launch an MCMC.

We work at the `build_info` layer rather than calling `quickstart` directly, so the configuration
stays an ordinary Cobaya `info` dict: the dataset to fit, the accuracy, and the cosmology are all
explicit, editable knobs. `build_info` still supplies the preset template and the Fiducial-map
parameter handling, and `resolve_aliases` recovers the role aliases (`.mflike`, `.lensing`) on the
model we build ourselves.

It builds on [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb), which produces the
cross-covariance file consumed in step 2.

> **ISO-local parameter overrides.** This notebook passes `params_dir="params"` to
> `build_info`, so cosmology defaults come from `ISO_sims/params/cosmo.yaml` (a copy of
> the packaged `soliket/presets/params/cosmo.yaml` you can edit here). Only `cosmo.yaml`
> is overridden — foreground and systematics fall back to the packaged defaults. Edit
> `params/cosmo.yaml` to change the ISO fiducial without touching the SOLikeT package.

## 1. A joint likelihood at the fiducial point

`build_info("multigaussian")` returns the `info` for a single `MultiGaussianLikelihood` that wires
MFLike + CMB lensing onto one shared CAMB theory. We set the packages path, expose the knobs worth
changing, build the model, and resolve the role aliases.

> **Neutrino sector — ISO override.** The preset default is cobaya's single-massive
> neutrino setup (`mnu = 0.06`, one massive eigenstate). This notebook overrides it to the
> SO **2-eigenstate normal hierarchy**: the `mnu*` params come from `params/cosmo.yaml`
> (kept fixed by `setdefault`, so they win over the injected default) and the matching camb
> `extra_args` are applied inline via `apply_nh(info)`. The override is **camb-only**.

In [ ]:
from cobaya.model import get_model
from cobaya.tools import resolve_packages_path

from soliket.presets import build_info, resolve_aliases

MGL = "soliket.gaussian.MultiGaussianLikelihood"
PACKAGES = resolve_packages_path()


def loglike(model):
    """Total log-likelihood of a model at its fiducial point."""
    return float(sum(model.loglikes({})[0]))


def apply_nh(info):
    """SO normal hierarchy (2 massive eigenstates), camb-only.

    The preset default is cobaya's single-massive neutrino setup; this overrides
    it for the ISO analysis. mnu is supplied by params/cosmo.yaml; the matching
    camb extra_args go here. Returns info (mutated in place) for chaining.
    """
    ea = info["theory"]["camb"]["extra_args"]
    ea.pop("num_massive_neutrinos", None)  # drop the injected single-massive default
    ea.pop("nnu", None)                    # Neff comes from num_nu_massless + share_delta_neff
    ea.update({
        "num_nu_massless": 1.044,
        "num_nu_massive": 2,
        "nu_mass_eigenstates": 2,
        "nu_mass_fractions": [0.14763410387308012, 0.8523658961269198],
        "nu_mass_numbers": [1, 1],
        "share_delta_neff": True,
    })
    return info


info = build_info("multigaussian", params_dir="params")
info["packages_path"] = PACKAGES
apply_nh(info)
# --- editable knobs --------------------------------------------------------
# info["likelihood"][MGL]["options"][0]["input_file"] = "my_sim.fits"        # data to fit
# info["theory"]["camb"]["extra_args"]["lens_potential_accuracy"] = 4        # accuracy
# ---------------------------------------------------------------------------

model = get_model(info)
roles = resolve_aliases(model)

print("members :", type(roles.mflike).__name__, "+", type(roles.lensing).__name__)
print("joint loglike at fiducial:", round(loglike(model), 4))

The two probes are treated as **independent** so far: the joint covariance is block-diagonal
(MFLike auto-covariance, lensing auto-covariance, and zeros off the diagonal).

## 2. Adding the cross-covariance

Because both probes look at the same sky, their errors are correlated. The physical cross-covariance
comes from [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb). `from_cmb_lensing` accepts
anything exposing the `.mflike` / `.lensing` roles, so the `AliasView` from `resolve_aliases` works
directly:

```python
from soliket.gaussian.gaussian_data import CrossCov

xcov = CrossCov.from_cmb_lensing(roles)   # full-accuracy CAMB derivative (~3 GB, minutes)
xcov.save("XCov_mflike_lensing.fits")
```

It keys the block by the components' real names and carries their auto-covariances, so the saved file
drops straight into the likelihood via `cross_cov_path`. That heavy derivative is not run here; the
cell below stands in a **small illustrative** cross-covariance with the same structure (real component
names + auto-covariances + an off-diagonal block) so we can see the wiring and its effect.

In [ ]:
import os
import tempfile

import numpy as np

from soliket.gaussian.gaussian_data import CrossCov

# The two component data objects (names + auto-covariances), via the role aliases.
mflike_data = roles.mflike._get_gauss_data()
lensing_data = roles.lensing._get_gauss_data()

# A small illustrative cross-covariance block; from_cmb_lensing fills this with the real,
# CAMB-derived CMB x lensing covariance instead.
rho = 1e-5
block = rho * np.outer(
    np.sqrt(np.diag(mflike_data.cov)), np.sqrt(np.diag(lensing_data.cov))
)

xcov = CrossCov()
xcov.add_component(mflike_data.name, mflike_data.cov)
xcov.add_component(lensing_data.name, lensing_data.cov)
xcov.add_cross_covariance(mflike_data.name, lensing_data.name, block)

xcov_path = os.path.join(tempfile.mkdtemp(prefix="soliket_an_"), "XCov_demo.fits")
xcov.save(xcov_path)
print("cross-covariance written to", xcov_path)

Pass the file to the likelihood through `cross_cov_path`. We rebuild from the same preset `info`
with that one extra option, and compare the joint log-likelihood with and without the cross term.

In [ ]:
info_xcov = build_info("multigaussian", params_dir="params")
info_xcov["packages_path"] = PACKAGES
apply_nh(info_xcov)
info_xcov["likelihood"][MGL]["cross_cov_path"] = xcov_path

model_xcov = get_model(info_xcov)

print("loglike without cross-cov:", round(loglike(model), 4))
print("loglike with    cross-cov:", round(loglike(model_xcov), 4))

## 3. Running an MCMC

Sampling is Python-native. Passing `sample=[...]` to `build_info` turns the named dual parameters into
sampled ones (they get their priors back); add a `sampler` block and hand the `info` to `cobaya.run`.
The cell below sets up a short chain over `tau`, guarded by `RUN_MCMC` so the notebook runs
top-to-bottom without launching a multi-minute sampler.

In [ ]:
RUN_MCMC = False  # set True to launch the sampler (minutes)

info_mcmc = build_info("multigaussian", sample=["tau"], params_dir="params")
info_mcmc["packages_path"] = PACKAGES
apply_nh(info_mcmc)

sampled = [p for p, v in info_mcmc["params"].items() if isinstance(v, dict) and "prior" in v]
print("sampled parameters:", sampled)

if RUN_MCMC:
    from cobaya import run

    info_mcmc["sampler"] = {"mcmc": {"max_samples": 50, "Rminus1_stop": 0.1}}
    info_mcmc["output"] = "chains/multigaussian"
    updated_info, sampler = run(info_mcmc)
    print("done:", sampler.products()["sample"].shape)
else:
    print("RUN_MCMC is False - skipping the sampler.")

## Recap

- `build_info("multigaussian")` gives the joint MFLike + CMB-lensing `info`; with the data file,
  accuracy and cosmology as explicit knobs. `resolve_aliases(model)` recovers `.mflike` / `.lensing`.
- The cross-covariance from [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) folds in
  through `cross_cov_path`; `CrossCov.from_cmb_lensing(roles)` produces a file that wires in directly.
- Sampling runs by adding a `sampler` block and calling `cobaya.run(info)`.

`quickstart("multigaussian")` does all of the above in one call when you want the convenience instead
of the knobs. For building the simulated inputs, see [`create_datasets.ipynb`](create_datasets.ipynb);
for the hand-built single-likelihood walkthrough,
[`../first_step_tutorial.ipynb`](../first_step_tutorial.ipynb).